# CPL Cleaning Pipeline

Reads raw data from S3, cleans branches and census tracts, writes clean CSVs back to S3.
Defined as a KFP pipeline so runs are traceable in the RHOAI dashboard.

In [ ]:
import os
import kfp
from kfp import dsl, kubernetes
from kfp.dsl import Output, Dataset

## Pipeline Components

In [ ]:
RUNTIME_IMAGE = "image-registry.openshift-image-registry.svc:5000/redhat-ods-applications/runtime-datascience:datascience"


@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3"],
    pip_index_urls=["https://pypi.org/simple"],
)
def clean_branches(branches_csv: Output[Dataset]):
    import ast
    import io
    import os
    import boto3
    import pandas as pd

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    raw_key = "raw/branches/Libraries_-_Locations___Contact_Information__and_Usual_Hours_of_Operation.csv"
    obj = s3.get_object(Bucket=bucket, Key=raw_key)
    df = pd.read_csv(io.BytesIO(obj["Body"].read()))

    df.columns = df.columns.str.strip()
    df = df.rename(columns={"NAME": "BRANCH"})

    name_fixes = {
        "Daley, Richard J.": "Daley, Richard J. - Bridgeport",
        "Daley, Richard M.": "Daley, Richard M. - W. Humboldt",
        "Harold Washtington Library Center": "Harold Washington Library Center",
        "King": "Martin Luther King Jr.",
    }
    df["BRANCH"] = df["BRANCH"].replace(name_fixes)
    df["PERMANENTLY_CLOSED"] = 0

    roosevelt = pd.DataFrame([{
        "BRANCH": "Roosevelt",
        "HOURS OF OPERATION": None,
        "ADDRESS": "1101 W. Taylor Street",
        "CITY": "Chicago", "STATE": "IL", "ZIP": "60607",
        "PHONE": None, "WEBSITE": None,
        "LOCATION": "(41.86943838326072, -87.6544586134924)",
        "PERMANENTLY_CLOSED": 1,
    }])
    df = pd.concat([df, roosevelt], ignore_index=True)

    coords = df["LOCATION"].apply(ast.literal_eval)
    df["LATITUDE"] = coords.apply(lambda c: float(c[0]))
    df["LONGITUDE"] = coords.apply(lambda c: float(c[1]))

    df = df.sort_values("BRANCH").reset_index(drop=True)

    df.to_csv(branches_csv.path, index=False)

    buf = io.BytesIO()
    df.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="clean/branches.csv", Body=buf.getvalue())
    print(f"Wrote {len(df)} branches to s3://{bucket}/clean/branches.csv")

In [ ]:
@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "geopandas", "pyogrio"],
    pip_index_urls=["https://pypi.org/simple"],
)
def clean_census_tracts(tracts_csv: Output[Dataset]):
    import io
    import os
    import tempfile
    import boto3
    import pandas as pd
    import geopandas as gpd

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    obj = s3.get_object(Bucket=bucket, Key="raw/boundaries/Boundaries - City.geojson")
    chicago = gpd.read_file(io.BytesIO(obj["Body"].read()))

    obj = s3.get_object(Bucket=bucket, Key="raw/boundaries/tl_2021_17_tract.zip")
    with tempfile.NamedTemporaryFile(suffix=".zip") as tmp:
        tmp.write(obj["Body"].read())
        tmp.flush()
        il_tracts = gpd.read_file(tmp.name)

    il_tracts = il_tracts.to_crs(chicago.crs)
    chicago_tracts = gpd.clip(il_tracts, chicago)

    obj = s3.get_object(Bucket=bucket, Key="raw/demographics/R13395758_SL140.csv")
    demos = pd.read_csv(io.BytesIO(obj["Body"].read()))
    demos = demos.iloc[1:, :]

    merged = chicago_tracts.merge(demos, how="left", left_on="GEOID", right_on="FIPS")

    id_cols = {
        "GEOID": "geoid",
        "Qualifying Name": "qualifying name",
        "ALAND": "current land area (sq. m)",
        "AWATER": "current water area (sq. m)",
    }
    merged = merged.rename(columns=id_cols)

    pop_cols = [c for c in merged.columns if c.startswith("Total Population") or
                c.startswith("Population Density") or c.startswith("Total:")]

    keep = list(id_cols.values()) + pop_cols + ["geometry"]
    keep = [c for c in keep if c in merged.columns]
    result = merged[keep].copy()
    result.columns = result.columns.str.lower()

    result.to_csv(tracts_csv.path, index=False)

    buf = io.BytesIO()
    result.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="clean/population_demographics.csv", Body=buf.getvalue())
    print(f"Wrote {len(result)} tracts to s3://{bucket}/clean/population_demographics.csv")

In [ ]:
@dsl.component(
    base_image=RUNTIME_IMAGE,
    packages_to_install=["boto3", "geopandas", "pyogrio"],
    pip_index_urls=["https://pypi.org/simple"],
)
def clean_census_demos(demos_csv: Output[Dataset]):
    import io
    import os
    import tempfile
    import boto3
    import pandas as pd
    import geopandas as gpd

    s3 = boto3.client(
        "s3",
        endpoint_url=os.environ["AWS_S3_ENDPOINT"],
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        verify=False,
    )
    bucket = os.environ["AWS_S3_BUCKET"]

    obj = s3.get_object(Bucket=bucket, Key="raw/boundaries/Boundaries - City.geojson")
    chicago = gpd.read_file(io.BytesIO(obj["Body"].read()))

    obj = s3.get_object(Bucket=bucket, Key="raw/boundaries/tl_2021_17_tract.zip")
    with tempfile.NamedTemporaryFile(suffix=".zip") as tmp:
        tmp.write(obj["Body"].read())
        tmp.flush()
        il_tracts = gpd.read_file(tmp.name)

    il_tracts = il_tracts.to_crs(chicago.crs)
    chicago_tracts = gpd.clip(il_tracts, chicago)

    obj = s3.get_object(Bucket=bucket, Key="raw/demographics/R13395758_SL140.csv")
    demos = pd.read_csv(io.BytesIO(obj["Body"].read()))
    demos = demos.iloc[1:, :]

    merged = chicago_tracts.merge(demos, how="left", left_on="GEOID", right_on="FIPS")

    keep_cols = ["GEOID", "Qualifying Name", "geometry", "Total Population"]
    pct_cols = [c for c in merged.columns if c.startswith("Percent") or c.startswith("average commute")]
    keep_cols.extend(pct_cols)
    keep_cols = [c for c in keep_cols if c in merged.columns]
    result = merged[keep_cols].copy()

    result = result.rename(columns={
        "GEOID": "geoid",
        "Qualifying Name": "qualifying name",
        "Total Population": "total population",
    })

    for col in result.columns:
        if col.startswith("Percent") or col.startswith("average"):
            result[col] = pd.to_numeric(result[col], errors="coerce")

    result.to_csv(demos_csv.path, index=False)

    buf = io.BytesIO()
    result.to_csv(buf, index=False)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key="clean/census_demos.csv", Body=buf.getvalue())
    print(f"Wrote {len(result)} tract demographics to s3://{bucket}/clean/census_demos.csv")

## Pipeline Definition

In [ ]:
@dsl.pipeline(
    name="cpl-cleaning",
    description="Clean raw CPL branch and census tract data for coverage analysis",
)
def cleaning_pipeline():
    branches_task = clean_branches()
    kubernetes.use_secret_as_env(
        branches_task,
        secret_name="cpl-s3",
        secret_key_to_env={
            "AWS_ACCESS_KEY_ID": "AWS_ACCESS_KEY_ID",
            "AWS_SECRET_ACCESS_KEY": "AWS_SECRET_ACCESS_KEY",
            "AWS_S3_ENDPOINT": "AWS_S3_ENDPOINT",
            "AWS_S3_BUCKET": "AWS_S3_BUCKET",
        },
    )

    tracts_task = clean_census_tracts()
    kubernetes.use_secret_as_env(
        tracts_task,
        secret_name="cpl-s3",
        secret_key_to_env={
            "AWS_ACCESS_KEY_ID": "AWS_ACCESS_KEY_ID",
            "AWS_SECRET_ACCESS_KEY": "AWS_SECRET_ACCESS_KEY",
            "AWS_S3_ENDPOINT": "AWS_S3_ENDPOINT",
            "AWS_S3_BUCKET": "AWS_S3_BUCKET",
        },
    )

    demos_task = clean_census_demos()
    kubernetes.use_secret_as_env(
        demos_task,
        secret_name="cpl-s3",
        secret_key_to_env={
            "AWS_ACCESS_KEY_ID": "AWS_ACCESS_KEY_ID",
            "AWS_SECRET_ACCESS_KEY": "AWS_SECRET_ACCESS_KEY",
            "AWS_S3_ENDPOINT": "AWS_S3_ENDPOINT",
            "AWS_S3_BUCKET": "AWS_S3_BUCKET",
        },
    )

## Submit

In [ ]:
import subprocess
from kfp.compiler import Compiler
from kfp_server_api.exceptions import ApiException

token = subprocess.check_output("oc whoami -t", shell=True, text=True).strip()
pipeline_url = os.environ.get("PIPELINES_URL")

client = kfp.Client(
    host=pipeline_url,
    existing_token=token,
    verify_ssl=False,
)

Compiler().compile(cleaning_pipeline, "cleaning_pipeline.yaml")

try:
    client.upload_pipeline("cleaning_pipeline.yaml", pipeline_name="cpl-cleaning")
    print("Pipeline definition registered.")
except ApiException as e:
    if e.status == 409:
        print("Pipeline definition already exists.")
    else:
        raise

run = client.create_run_from_pipeline_func(
    cleaning_pipeline,
    experiment_name="cpl-cleaning",
    run_name="cpl-cleaning-run",
    enable_caching=False,
)
print(f"Pipeline run submitted: {run.run_id}")